# Animal-Free Toxicology: NAMs, Organoids & Computational Methods
### 3Rs · Organ-on-Chip · Organoids · Digital Twins · AI-Driven Hazard Assessment

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

## The 3Rs: Replace, Reduce, Refine

The 3Rs (Russell & Burch, 1959) are the ethical and scientific framework for humane science. They are now **legally mandated** in the EU (Directive 2010/63/EU), the US (ICCVAM Authorization Act, FDA Modernization Act 2.0 — 2022), and increasingly required by regulators (EPA, FDA, OECD).

```
REPLACE                         REDUCE                          REFINE
──────────────────────          ──────────────────────          ──────────────────────
QSAR / in silico prediction     Tiered testing strategies       Humane endpoints
Organ-on-chip / organoids       Read-across (no new tests)      Anaesthesia protocols
High-throughput cell assays     PBPK (fewer animals needed)     Biomarker endpoints
Human iPSC-derived models       Statistical power analysis      Less-invasive sampling
AI/ML tox prediction            Multi-endpoint assays           Enriched housing
```

## Why this is happening NOW

| Driver | Impact |
|--------|--------|
| **FDA Modernization Act 2.0 (2022)** | Drug approval no longer requires animal data — NAMs accepted |
| **EPA New Approach Methods Work Plan 2021–2025** | 30% reduction in vertebrate tests by 2025, 100% by 2035 |
| **EU Chemicals Strategy for Sustainability** | Mandatory transition to NAMs for REACH by 2030 |
| **OECD TG 497 (2023)** | First validated Defined Approach — no animal needed for skin sensitisation |
| **NIH 3Rs Policy 2023** | All grant applications must justify animal use |
| **Organoid revolution** | Patient-derived organoids now match in vivo drug response (>85% concordance) |

## What you will learn

| Section | Technology | Replaces |
|---------|-----------|---------|
| 1. Animal-free data landscape | Databases, ToxCast, CCTE | LD50, repeated dose |
| 2. High-throughput cytotoxicity | AC50, cytotox correction | MTD animal studies |
| 3. Organ-on-chip modelling | OoC data simulation & analysis | Multi-organ repeat dose |
| 4. Organoid toxicology | Liver, intestinal, brain organoids | Organ-specific animal tests |
| 5. iPSC-derived cardiomyocytes | hiPS-CM MEA + impedance | hERG + in vivo QT |
| 6. Microphysiological systems | Multi-organ MPS scoring | Systemic toxicity |
| 7. Digital twin toxicology | PBPK + QSAR + MPS fusion | Long-term animal studies |
| 8. AI prediction with uncertainty | Conformal prediction, UQ | Screening animal tests |
| 9. Power analysis & 3Rs design | Statistical dose spacing | Reduce animal numbers |
| 10. Regulatory submission (3Rs) | IATA WoE report | Full animal test battery |

---
## Section 1 — Animal-Free Data Landscape

Before running any new assay, check what data already exists. Many animal tests can be avoided using curated databases.

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
# !pip install rdkit scikit-learn xgboost shap matplotlib seaborn pandas scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')
import json
from dataclasses import dataclass, field
from typing import Optional
from scipy import stats, optimize, signal
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED, DataStructs
from rdkit.Chem.MolStandardize import rdMolStandardize

np.random.seed(42)
print("Imports OK ✓")

In [ ]:
# ── 1.1 The animal-free data hierarchy ────────────────────────────────────────
# Before requesting any experiment, consult these sources in order.
# This alone often avoids new animal experiments.

ANIMAL_FREE_DATABASES = {
    "ToxCast / Comptox": {
        "url":        "https://comptox.epa.gov/dashboard/",
        "n_chemicals":  ~10_000,
        "assays":       9_000,
        "content":      "HTS assay data, bioactivity, physicochemical, QSAR predictions",
        "replaces":     "Initial hazard screening",
        "api":          "comptox.epa.gov/dashboard/api",
    },
    "ChemBL": {
        "url":        "https://www.ebi.ac.uk/chembl/",
        "n_chemicals":  ~2_400_000,
        "assays":       1_400_000,
        "content":      "Bioactivity, ADMET, drug targets, clinical data",
        "replaces":     "Target-based toxicity screening",
        "api":          "chembl.ebi.ac.uk/chembl/api/data/",
    },
    "ECOTOX": {
        "url":        "https://cfpub.epa.gov/ecotox/",
        "n_chemicals":  13_000,
        "content":      "Aquatic/terrestrial ecotoxicity, species sensitivity",
        "replaces":     "Ecotox animal tests (fish, daphnia)",
    },
    "TOXNET / HSDB": {
        "url":        "https://www.ncbi.nlm.nih.gov/pcsubstance/",
        "content":      "Human health data, chemical hazard summaries",
        "replaces":     "Literature animal data",
    },
    "Tox21": {
        "url":        "https://tox21.gov/",
        "n_chemicals":  12_000,
        "assays":       71,
        "content":      "Nuclear receptor, stress pathway, cell viability",
        "replaces":     "Mechanistic in vivo studies",
    },
    "Open TG-GATEs": {
        "url":        "https://toxico.nibiohn.go.jp/english/",
        "n_chemicals":  170,
        "content":      "Transcriptomics (rat, human hepatocytes), histopath, clinical chem",
        "replaces":     "28-day repeat dose — already done",
    },
    "DrugMatrix": {
        "url":        "https://ntp.niehs.nih.gov/data/drugmatrix/",
        "n_chemicals":  600,
        "content":      "Rat organ transcriptomics after in vivo dosing",
        "replaces":     "Organ-specific repeat-dose — existing data",
    },
    "DILIrank": {
        "url":        "https://www.fda.gov/science-research/",
        "n_chemicals":  1_036,
        "content":      "FDA DILI severity classification (Most/Less/No concern)",
        "replaces":     "Hepatotoxicity animal studies",
    },
}

print(f"{'Database':25s} {'Compounds':>12} {'What it replaces'}")
print("-" * 70)
for db, info in ANIMAL_FREE_DATABASES.items():
    n = info.get("n_chemicals", "varies")
    n_str = f"~{n:,}" if isinstance(n, int) else str(n)
    print(f"{db:25s} {n_str:>12}   {info['replaces']}")

In [ ]:
# ── 1.2 Simulate ToxCast HTS data retrieval ──────────────────────────────────
# In production: use EPA CompTox API or R toxcastR package
# Here we simulate the data structure and analysis workflow.

def simulate_toxcast_profile(smiles: str, seed: int = 42) -> pd.DataFrame:
    """
    Simulate a ToxCast assay activity profile for a compound.
    Real usage: requests.get('https://comptox.epa.gov/dashboard/api/...')
    """
    np.random.seed(seed + hash(smiles) % 1000)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return pd.DataFrame()

    logp  = Descriptors.MolLogP(mol)
    mw    = Descriptors.MolWt(mol)

    # ToxCast assay categories (representative subset)
    assay_categories = {
        "Nuclear receptors":    ["AR_agonist","ER_agonist","AhR","PPARg","GR","TR_alpha","RAR"],
        "Cell stress":          ["NRF2","HSP70","p53","ATM","GADD45","OxStress"],
        "Cell viability":       ["CellTox_Green","LDH_release","ATP_content","MitoMembrane"],
        "Kinase inhibition":    ["EGFR","MAPK","PI3K","CDK2","JAK2"],
        "Transporter":          ["BSEP_inhib","P-gp","MRP2","OATP1B1"],
    }

    records = []
    for category, assays in assay_categories.items():
        for assay in assays:
            # Simulate activity probability from physicochemical properties
            base_prob = 0.1 + 0.05*max(0, logp-2) + 0.0005*mw
            if "CellTox" in assay or "LDH" in assay:
                base_prob += 0.1*max(0, logp-3)
            if "BSEP" in assay:
                base_prob += 0.08*(mw > 400)

            active = np.random.rand() < np.clip(base_prob, 0, 0.85)
            ac50   = np.random.lognormal(1.5, 1.2) if active else None

            records.append({
                "assay_name": assay,
                "category":   category,
                "active":     int(active),
                "AC50_uM":    round(float(ac50), 3) if ac50 else None,
                "hitc":       int(active),  # ToxCast hit-call
            })

    return pd.DataFrame(records)

# Profile two compounds
aspirin_profile = simulate_toxcast_profile("CC(=O)Oc1ccccc1C(=O)O", seed=1)
bpa_profile     = simulate_toxcast_profile("CC(C)(c1ccc(O)cc1)c1ccc(O)cc1", seed=2)

print("ToxCast HTS Profile Summary:")
for name, df in [("Aspirin", aspirin_profile), ("Bisphenol A", bpa_profile)]:
    n_active = df["active"].sum()
    n_total  = len(df)
    print(f"\n  {name}: {n_active}/{n_total} assays active ({n_active/n_total*100:.0f}%)")
    by_cat = df.groupby("category")["active"].agg(["sum","count"])
    for cat, row in by_cat.iterrows():
        bar = "█" * int(row["sum"])
        print(f"    {cat:22s}: {int(row['sum']):2d}/{int(row['count'])} active  {bar}")
    if df["active"].any():
        print(f"    Min AC50 (active): {df[df['active']==1]['AC50_uM'].min():.2f} μM")

---
## Section 2 — High-Throughput Cytotoxicity & AC50 Analysis

HTS cytotoxicity assays (ToxCast, Tox21) provide concentration-response data that replaces acute toxicity animal tests when properly analysed with cytotoxicity correction.

In [ ]:
# ── 2.1 Cytotoxicity-corrected potency (burst ratio method) ──────────────────
# Problem: many HTS 'actives' are artefacts of cytotoxicity, not specific activity.
# Solution: cytotoxicity correction — flag hits where AC50_bioactivity ≥ AC50_cytotox
# Reference: Judson 2016, Watt 2018 (EPA/Tox21 burst ratio method)

def hill_4p(conc, ec50, hill, top, bottom):
    """4-parameter Hill equation."""    return bottom + (top - bottom) / (1 + (ec50 / conc) ** hill)

def fit_concentration_response(concs: np.ndarray,
                                responses: np.ndarray,
                                name: str = "") -> dict:
    """Fit Hill equation to concentration-response data. Returns AC50 and fit quality."""    try:
        popt, pcov = optimize.curve_fit(
            hill_4p, concs, responses,
            p0=[np.median(concs), 1.0, responses.max(), responses.min()],
            bounds=([concs.min()*0.001, 0.1, 0, -20],
                    [concs.max()*1000,   10,  200, 20]),
            maxfev=5000
        )
        perr = np.sqrt(np.diag(pcov))
        pred = hill_4p(concs, *popt)
        ss_res = ((responses - pred)**2).sum()
        ss_tot = ((responses - responses.mean())**2).sum()
        r2 = 1 - ss_res / ss_tot

        return {"ec50": popt[0], "hill": popt[1], "top": popt[2], "bottom": popt[3],
                "r2": r2, "ec50_se": perr[0], "converged": True, "popt": popt}
    except Exception:
        return {"ec50": None, "converged": False}

def cytotox_correction(bioactivity_ac50: Optional[float],
                        cytotox_ac50: Optional[float],
                        burst_ratio_threshold: float = 0.1) -> dict:
    """
    Flag hits where bioactivity AC50 ≥ burst_ratio × cytotox AC50.
    If the biological effect occurs at cytotoxic concentrations → likely artefact.

    burst_ratio_threshold = 0.1 means: flag if bio_AC50 ≥ 10% of cytotox_AC50
    Reference: Judson et al. 2016 (Chem Res Toxicol)
    """
    if bioactivity_ac50 is None:
        return {"call": "INACTIVE", "flag": False, "reason": "No fit"}
    if cytotox_ac50 is None:
        return {"call": "ACTIVE_UNCONFIRMED", "flag": True,
                "reason": "No cytotox data — cannot correct"}

    ratio = bioactivity_ac50 / cytotox_ac50
    if ratio >= burst_ratio_threshold:
        return {"call": "CYTOTOX_ARTEFACT", "flag": True,
                "ratio": round(ratio, 3),
                "reason": f"Bio AC50 ({bioactivity_ac50:.2f} μM) ≥ {burst_ratio_threshold:.0%} × cytotox ({cytotox_ac50:.2f} μM)"}
    else:
        return {"call": "ACTIVE_SPECIFIC", "flag": False,
                "ratio": round(ratio, 3),
                "reason": f"Bio AC50 well below cytotox threshold (ratio={ratio:.3f})"}

# Simulate HTS plate data: 10-point concentration-response
np.random.seed(42)
concs_hts = np.logspace(-3, 2, 10)  # 0.001 to 100 μM

# Compound A: specific activator (AC50 = 1 μM, cytotox = 50 μM)
y_bio_A  = hill_4p(concs_hts, 1.0,  1.5, 80, 2)  + np.random.normal(0, 5, 10)
y_tox_A  = hill_4p(concs_hts, 50.0, 2.0, 90, 3)  + np.random.normal(0, 4, 10)

# Compound B: cytotoxicity artefact (both AC50 similar = 8 μM)
y_bio_B  = hill_4p(concs_hts, 8.0,  1.8, 75, 3)  + np.random.normal(0, 5, 10)
y_tox_B  = hill_4p(concs_hts, 10.0, 2.5, 88, 2)  + np.random.normal(0, 4, 10)

for cpd, y_bio, y_tox in [("Compound A (specific)", y_bio_A, y_tox_A),
                            ("Compound B (artefact)",  y_bio_B, y_tox_B)]:
    bio_fit  = fit_concentration_response(concs_hts, y_bio)
    tox_fit  = fit_concentration_response(concs_hts, y_tox)
    cytocorr = cytotox_correction(bio_fit.get("ec50"), tox_fit.get("ec50"))
    print(f"{cpd}")
    print(f"  Bioactivity AC50: {bio_fit.get('ec50', 'N/A'):.3f} μM  (R²={bio_fit.get('r2',0):.3f})")
    print(f"  Cytotox AC50:     {tox_fit.get('ec50', 'N/A'):.3f} μM")
    print(f"  → {cytocorr['call']}: {cytocorr['reason']}")
    print()

---
## Section 3 — Organ-on-Chip Data Analysis

Organ-on-a-chip (OoC) systems contain living human cells in microfluidic devices that mimic organ physiology. They provide ADME and toxicity data that directly replaces animal experiments.

In [ ]:
# ── 3.1 OoC experimental design and data structure ────────────────────────────
# Key OoC platforms: Emulate, Mimetas (OrganoPlate), CN Bio, TissUse (HUMIMIC)
# Typical readouts: barrier integrity (TEER), cytokine secretion, metabolomics,
#                   imaging (live/dead, nuclear morphology)

@dataclass
class OoCExperiment:
    """
    Represents a single Organ-on-Chip toxicity experiment.
    Structure follows Microphysiological Systems (MPS) data reporting standards
    (NIH MPS Database, CIPA MPS working group).
    """
    organ_type:    str    # e.g. "liver", "gut", "kidney", "lung", "brain"
    platform:      str    # e.g. "Emulate", "CN Bio PhysioMimix", "Mimetas"
    cell_source:   str    # "primary", "iPSC-derived", "HepaRG", "Caco-2"
    compound:      str
    smiles:        str
    concentration_uM: float
    exposure_days: int
    readouts:      dict   # measured outputs

@dataclass
class OoCEndpoint:
    name:        str
    unit:        str
    baseline:    float   # vehicle control value
    threshold:   float   # toxicity threshold (% change from baseline)
    direction:   str     # "decrease_is_toxic" or "increase_is_toxic"

# ── Standard OoC endpoints by organ type ──────────────────────────────────────
OOC_ENDPOINTS = {
    "liver": [
        OoCEndpoint("TEER",            "Ω·cm²",  250.0, -20.0, "decrease_is_toxic"),
        OoCEndpoint("albumin_secretion","μg/mL/d", 15.0, -30.0, "decrease_is_toxic"),
        OoCEndpoint("ALT_release",     "U/L",      5.0,  100.0, "increase_is_toxic"),
        OoCEndpoint("ATP_content",     "% ctrl",  100.0, -20.0, "decrease_is_toxic"),
        OoCEndpoint("urea_synthesis",  "μg/mL/d", 12.0,  -25.0, "decrease_is_toxic"),
        OoCEndpoint("CYP3A4_activity", "% ctrl",  100.0, -30.0, "decrease_is_toxic"),
        OoCEndpoint("ROS_production",  "% ctrl",  100.0,  50.0, "increase_is_toxic"),
        OoCEndpoint("mitochondria_mem","% ctrl",  100.0, -25.0, "decrease_is_toxic"),
    ],
    "gut": [
        OoCEndpoint("TEER",            "Ω·cm²",  400.0, -20.0, "decrease_is_toxic"),
        OoCEndpoint("permeability_FD4","cm/s",     2e-7,  100.0,"increase_is_toxic"),
        OoCEndpoint("villin_expression","% ctrl", 100.0,  -25.0,"decrease_is_toxic"),
        OoCEndpoint("mucus_thickness", "μm",       20.0,  -30.0,"decrease_is_toxic"),
        OoCEndpoint("cytokine_IL8",    "pg/mL",    50.0,  200.0,"increase_is_toxic"),
    ],
    "kidney": [
        OoCEndpoint("creatinine_clear","mL/min",    1.2, -25.0, "decrease_is_toxic"),
        OoCEndpoint("KIM1_release",    "ng/mL",     0.5, 200.0, "increase_is_toxic"),
        OoCEndpoint("NGAL_release",    "ng/mL",     2.0, 150.0, "increase_is_toxic"),
        OoCEndpoint("cell_viability",  "% ctrl",  100.0, -20.0, "decrease_is_toxic"),
    ],
}

def simulate_ooc_experiment(smiles: str, organ: str,
                              concentration_uM: float,
                              exposure_days: int = 7,
                              seed: int = 42) -> OoCExperiment:
    """Simulate OoC readouts for a compound."""    np.random.seed(seed + int(concentration_uM * 10))
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")

    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)
    # Toxicity proxy: based on LogP and concentration
    tox_score = (logp/5) * (np.log10(concentration_uM + 1) / 3)

    endpoints = OOC_ENDPOINTS.get(organ, [])
    readouts  = {}
    for ep in endpoints:
        noise   = np.random.normal(0, 0.08)
        if ep.direction == "decrease_is_toxic":
            val = ep.baseline * (1 - tox_score * np.random.uniform(0.5, 1.5) + noise)
        else:
            val = ep.baseline * (1 + tox_score * np.random.uniform(1, 3) + abs(noise))
        readouts[ep.name] = {"value": round(float(val), 3), "unit": ep.unit,
                              "baseline": ep.baseline}

    return OoCExperiment(
        organ_type=organ, platform="CN Bio PhysioMimix",
        cell_source="iPSC-derived" if organ=="liver" else "primary",
        compound=smiles[:20], smiles=smiles,
        concentration_uM=concentration_uM, exposure_days=exposure_days,
        readouts=readouts
    )

def score_ooc_toxicity(experiment: OoCExperiment) -> dict:
    """
    Score OoC experiment against endpoint thresholds.
    Returns per-endpoint calls and overall organ toxicity score.
    """
    organ    = experiment.organ_type
    endpoints = {ep.name: ep for ep in OOC_ENDPOINTS.get(organ, [])}
    n_toxic, n_total = 0, 0
    ep_calls = {}

    for ep_name, data in experiment.readouts.items():
        if ep_name not in endpoints:
            continue
        ep   = endpoints[ep_name]
        val  = data["value"]
        base = ep.baseline
        pct_change = (val - base) / base * 100

        if ep.direction == "decrease_is_toxic":
            toxic = pct_change <= ep.threshold
        else:
            toxic = pct_change >= ep.threshold

        ep_calls[ep_name] = {
            "value": val, "baseline": base,
            "pct_change": round(pct_change, 1),
            "toxic": toxic,
        }
        n_toxic += int(toxic)
        n_total += 1

    toxicity_index = n_toxic / n_total if n_total > 0 else 0
    severity = ("SEVERE" if toxicity_index > 0.6
                else "MODERATE" if toxicity_index > 0.3
                else "MILD"     if toxicity_index > 0.1
                else "NONE")

    return {
        "organ":          organ,
        "concentration":  experiment.concentration_uM,
        "toxicity_index": round(toxicity_index, 3),
        "severity":       severity,
        "n_toxic_endpoints": n_toxic,
        "n_total_endpoints": n_total,
        "endpoint_calls": ep_calls,
    }

# Test diclofenac (hepatotoxic NSAID) vs aspirin
for name, smi, conc in [
    ("Diclofenac (hepatotoxic)", "O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl", 50.0),
    ("Aspirin (control)",         "CC(=O)Oc1ccccc1C(=O)O",           50.0),
]:
    exp   = simulate_ooc_experiment(smi, "liver", conc, seed=hash(name)%1000)
    score = score_ooc_toxicity(exp)
    print(f"\n{name} — Liver OoC @ {conc} μM ({exp.cell_source})")
    print(f"  Severity: {score['severity']}  (Toxicity Index = {score['toxicity_index']:.2f})")
    print(f"  Toxic endpoints: {score['n_toxic_endpoints']}/{score['n_total_endpoints']}")
    for ep, data in score["endpoint_calls"].items():
        flag = "⚠" if data["toxic"] else " "
        print(f"    {flag} {ep:22s}: {data['value']:8.2f}  ({data['pct_change']:+.1f}% vs baseline)")

In [ ]:
# ── 3.2 Multi-concentration OoC dose-response ─────────────────────────────────
# Build a full dose-response curve using OoC readouts
# This replaces an in vivo MTD or NOAEL study

def ooc_dose_response(smiles: str, organ: str,
                       concentrations: np.ndarray,
                       key_endpoint: str = "ATP_content") -> pd.DataFrame:
    """Build dose-response from OoC experiments across concentration range."""    records = []
    for conc in concentrations:
        exp   = simulate_ooc_experiment(smiles, organ, float(conc), seed=int(conc*10))
        score = score_ooc_toxicity(exp)
        ep_data = score["endpoint_calls"].get(key_endpoint, {})
        pct_val = ep_data.get("pct_change", 0)
        records.append({
            "concentration_uM": conc,
            "toxicity_index":   score["toxicity_index"],
            "endpoint_pct":     100 + pct_val,  # convert to % of baseline
            "severity":         score["severity"],
        })
    return pd.DataFrame(records)

concs = np.logspace(-2, 3, 10)  # 0.01 to 1000 μM
chemicals = {
    "Diclofenac":  "O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl",
    "Troglitazone":"Cc1ccc(CC2SC(=O)NC2=O)cc1OCC(C)(C)c1ccc(O)cc1",
    "Metformin":   "CN(C)C(=N)NC(=N)N",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colours = {"Diclofenac": "#E74C3C", "Troglitazone": "#E67E22", "Metformin": "#1565C0"}
for name, smi in chemicals.items():
    df = ooc_dose_response(smi, "liver", concs)
    axes[0].semilogx(df["concentration_uM"], df["endpoint_pct"],
                     "o-", color=colours[name], lw=2.2, ms=7, label=name)
    axes[1].semilogx(df["concentration_uM"], df["toxicity_index"],
                     "o-", color=colours[name], lw=2.2, ms=7, label=name)

axes[0].axhline(80, color="k", linestyle="--", lw=1.2, alpha=0.5, label="80% threshold (IC20)")
axes[0].set_xlabel("Concentration (μM)"); axes[0].set_ylabel("ATP Content (% baseline)")
axes[0].set_title("Liver OoC — Viability Dose-Response", fontweight="bold")
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

axes[1].axhline(0.3, color="k", linestyle="--", lw=1.2, alpha=0.5, label="Moderate threshold")
axes[1].set_xlabel("Concentration (μM)"); axes[1].set_ylabel("Toxicity Index (0–1)")
axes[1].set_title("Liver OoC — Multi-Endpoint Toxicity Index", fontweight="bold")
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

plt.suptitle("Organ-on-Chip Liver Toxicity — Replaces 28-Day Rat Hepatotoxicity Study",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

---
## Section 4 — Organoid Toxicology

Organoids are self-organising 3D structures derived from stem cells that recapitulate organ architecture and function. They are the closest in vitro model to intact organs.

In [ ]:
# ── 4.1 Organoid model types and endpoints ────────────────────────────────────

ORGANOID_MODELS = {
    "Liver (hepatocyte)": {
        "cell_source":   "iPSC-derived hepatocytes or primary PHH",
        "culture":       "3D spheroid, 400–600 μm diameter",
        "stability":     "4–6 weeks functional",
        "endpoints":     ["albumin", "urea", "CYP3A4", "CYP1A2", "AST/ALT",
                          "BSEP function", "bile acid transport", "steatosis score"],
        "advantages":    "Zone-specific metabolism, bile canaliculi, long-term stability",
        "replaces":      "ICH S2, S7, hepatotox animal studies, 28-day rat liver",
        "platform":      "InSphero GravityPLUS, Lonza 3D InSight",
    },
    "Intestinal (enteroid)": {
        "cell_source":   "Crypt-derived organoids (LGR5+ stem cells) or Caco-2 3D",
        "culture":       "Matrigel embedded or air-liquid interface",
        "stability":     "Unlimited (can passage)",
        "endpoints":     ["TEER", "permeability P_app", "mucus secretion",
                          "secretory IgA", "microbiome co-culture", "inflammatory cytokines"],
        "advantages":    "Villus-crypt architecture, goblet cells, enteroendocrine cells",
        "replaces":      "GI tolerance animal studies, oral bioavailability estimate",
        "platform":      "Hubrecht Organoid Technology, STEMCELL TechnologIES",
    },
    "Brain (cerebral)": {
        "cell_source":   "iPSC-derived neural progenitors",
        "culture":       "Self-organising, 1–3 mm diameter",
        "stability":     "Months to years",
        "endpoints":     ["MEA firing rate", "network synchrony", "neural cell types",
                          "BBB permeability (vascularised)", "ROS", "apoptosis"],
        "advantages":    "Cortical layers, neuron subtypes, network activity",
        "replaces":      "DNT Tier 1 (ICH S5), developmental neurotoxicity studies",
        "platform":      "StemCell Technologies STEMdiff, Lancaster lab protocol",
    },
    "Kidney (tubuloid)": {
        "cell_source":   "Primary tubular cells or urine-derived iPSC",
        "culture":       "Matrigel 3D or microfluidic",
        "stability":     "8–12 weeks",
        "endpoints":     ["creatinine clearance", "cisplatin-induced KIM1",
                          "transporter expression", "OAT1/OCT2 function"],
        "advantages":    "Proximal tubule architecture, organic anion transporters",
        "replaces":      "Nephrotox rat studies, cisplatin kidney damage model",
        "platform":      "Hubrecht Institute, Mimetas OrganoPlate",
    },
    "Cardiac (cardioid)": {
        "cell_source":   "iPSC-derived cardiomyocytes (hiPS-CM)",
        "culture":       "3D beating spheroid or chamber-like",
        "stability":     "Weeks to months",
        "endpoints":     ["MEA field potential", "contractility (ImpedanceLive)",
                          "Ca2+ transients", "APD90", "EFP duration", "arrhythmia events"],
        "advantages":    "Human-relevant ion channels, beating physiology, CiPA endpoint",
        "replaces":      "hERG + ICH S7B in vivo QT study",
        "platform":      "Axiogenesis, Cellular Dynamics International, Ncardia Cor.4U",
    },
}

print("Organoid Models for Regulatory Toxicology")
print("=" * 70)
for model, info in ORGANOID_MODELS.items():
    print(f"\n{model}")
    print(f"  Source:    {info['cell_source']}")
    print(f"  Stability: {info['stability']}")
    print(f"  Replaces:  {info['replaces']}")
    print(f"  Key endpoints: {', '.join(info['endpoints'][:4])}...")

In [ ]:
# ── 4.2 Liver organoid viability + function scoring ───────────────────────────
# Models the InSphero liver spheroid assay (most validated for regulatory use).
# OECD validation study ongoing (2022–2025); FDA/EMA pilot submissions accepted.

def simulate_liver_organoid(smiles: str, concentration_uM: float,
                              exposure_days: int = 14,
                              seed: int = 42) -> dict:
    """
    Simulate liver organoid (3D hepatocyte spheroid) readouts.
    Based on InSphero GravityPLUS protocol + DILI benchmark compounds.
    """
    np.random.seed(seed)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {}

    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)
    naro = rdMolDescriptors.CalcNumAromaticRings(mol)

    # Tox score: lipophilic, aromatic compounds more hepatotoxic
    tox = np.clip((logp/5 + naro/4) * (np.log10(concentration_uM+1)/3), 0, 1)
    tox += np.random.normal(0, 0.05)

    return {
        # Viability
        "ATP_content_pct":     max(5, round(100*(1 - 0.8*tox) + np.random.normal(0,3), 1)),
        "LDH_leakage_pct":     max(0, round(100*0.6*tox + np.random.normal(0,4), 1)),
        "CellTiterGlo_pct":    max(5, round(100*(1 - 0.9*tox) + np.random.normal(0,3), 1)),
        # Function
        "albumin_ug_mL_d":     max(0, round(20*(1 - 0.7*tox) + np.random.normal(0,1.5), 2)),
        "urea_ug_mL_d":        max(0, round(15*(1 - 0.65*tox) + np.random.normal(0,1.2), 2)),
        "CYP3A4_act_pct":      max(0, round(100*(1 - 0.85*tox) + np.random.normal(0,6), 1)),
        "CYP1A2_act_pct":      max(0, round(100*(1 - 0.7*tox)  + np.random.normal(0,6), 1)),
        # Injury biomarkers
        "ALT_U_L":             max(0, round(8*tox*15 + np.random.normal(0,2), 1)),
        "AST_U_L":             max(0, round(12*tox*12 + np.random.normal(0,2), 1)),
        "HMGB1_ng_mL":         max(0, round(0.5*tox*20 + np.random.normal(0,0.5), 2)),
        # Mechanism markers
        "ROS_fold_ctrl":       max(1, round(1 + 4*tox + np.random.normal(0,0.3), 2)),
        "GSH_depletion_pct":   max(0, round(70*tox + np.random.normal(0,5), 1)),
        "mitochondria_JC1":    max(10,round(100*(1 - 0.75*tox) + np.random.normal(0,4),1)),
        "steatosis_area_pct":  max(0, round(30*tox + np.random.normal(0,3), 1)),
        # Metadata
        "exposure_days":       exposure_days,
        "concentration_uM":    concentration_uM,
    }

def classify_dili_risk(organoid_data: dict) -> dict:
    """
    DILI risk classification from liver organoid data.
    Thresholds from: Proctor 2017 (Drug Metab Dispos), Vorrink 2018 (ALTEX).
    """
    d = organoid_data
    concerns = []

    if d.get("ATP_content_pct", 100)     < 70:  concerns.append("ATP depletion < 70%")
    if d.get("LDH_leakage_pct", 0)       > 20:  concerns.append("LDH leakage > 20%")
    if d.get("albumin_ug_mL_d", 20)      < 12:  concerns.append("Albumin synthesis < 60%")
    if d.get("CYP3A4_act_pct", 100)      < 50:  concerns.append("CYP3A4 inhibition > 50%")
    if d.get("ROS_fold_ctrl", 1)         >  2:  concerns.append("ROS > 2× control")
    if d.get("GSH_depletion_pct", 0)     > 30:  concerns.append("GSH depletion > 30%")
    if d.get("mitochondria_JC1", 100)    < 60:  concerns.append("Mitochondria membrane loss")
    if d.get("steatosis_area_pct", 0)    > 15:  concerns.append("Steatosis > 15% area")

    n = len(concerns)
    dili_risk = "HIGH" if n >= 4 else "MODERATE" if n >= 2 else "LOW" if n >= 1 else "NONE"
    return {"DILI_risk": dili_risk, "n_concerns": n, "concerns": concerns}

# Screen a DILI benchmark panel
print("Liver Organoid DILI Screening (14-day exposure, 10 μM)")
print("="*70)
dili_panel = [
    ("Trovafloxacin  (HIGH DILI)",  "O=C(O)c1cn2c(=O)c(CN3CC(F)C3)cc2c2ccc(F)cc12",     10.0),
    ("Troglitazone   (HIGH DILI)",  "Cc1ccc(CC2SC(=O)NC2=O)cc1OCC(C)(C)c1ccc(O)cc1",    10.0),
    ("Diclofenac     (MOD DILI)",   "O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl",                    10.0),
    ("Metformin      (NO DILI)",    "CN(C)C(=N)NC(=N)N",                                  10.0),
    ("Penicillin G   (NO DILI)",    "CC1(C)SC2C(NC1=O)C(=O)N2Cc1ccccc1",                 10.0),
]
for name, smi, conc in dili_panel:
    data = simulate_liver_organoid(smi, conc, exposure_days=14, seed=hash(name)%2000)
    risk = classify_dili_risk(data)
    print(f"  {name:32s}: {risk['DILI_risk']:8s}  ({risk['n_concerns']} concerns)")
    if risk["concerns"]:
        print(f"    {', '.join(risk['concerns'][:3])}")

---
## Section 5 — hiPS-CM MEA: Replacing hERG + In Vivo QT Studies

Human iPSC-derived cardiomyocytes (hiPS-CM) on Multi-Electrode Arrays (MEA) are the CiPA Tier 2 assay. They provide human-relevant cardiac electrophysiology data that directly replaces the rabbit in vivo QT study.

In [ ]:
# ── 5.1 hiPS-CM MEA analysis pipeline ────────────────────────────────────────
# MEA records extracellular field potentials (EFP) from beating cardiomyocytes.
# Key endpoints: FPDc (corrected field potential duration) ≈ in vivo QTc
# Reference: Gintant 2016 (Nat Rev Drug Discov), Fermini 2016 (Nat Rev Drug Discov)

def simulate_hipscm_mea(smiles: str, concentration_uM: float,
                          seed: int = 42) -> dict:
    """
    Simulate hiPS-CM MEA electrophysiology readouts.
    Based on CiPA multi-site validation study (Gintant 2017).
    """
    np.random.seed(seed)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {}

    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)
    basic_n = sum(1 for a in mol.GetAtoms()
                  if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
    naro    = rdMolDescriptors.CalcNumAromaticRings(mol)

    # hERG-like activity proxy
    herg_prop = np.clip(0.25*logp + 0.15*basic_n + 0.08*naro - 0.5, 0, 1)
    herg_prop += np.random.normal(0, 0.08)

    # Baseline (vehicle)
    fpd_base  = np.random.normal(380, 15)   # ms — FPD baseline
    bpm_base  = np.random.normal(55, 5)     # beats per minute

    # Drug effect
    fpd_drug  = fpd_base  * (1 + 0.5*herg_prop)   # IKr block → FPD prolongation
    bpm_drug  = bpm_base  * (1 - 0.3*herg_prop)   # negative chronotropy
    amplitude = 100 * (1 - 0.4*herg_prop)          # reduced amplitude

    # Arrhythmia probability
    p_ead = np.clip(herg_prop**2 * 2, 0, 0.95)
    arrhythmia = np.random.rand() < p_ead

    # Bazett-corrected FPD (FPDc = FPD / sqrt(RR))
    rr_ms      = 60000 / bpm_drug                  # RR interval in ms
    fpdc_drug  = fpd_drug / np.sqrt(rr_ms / 1000)
    fpdc_base  = fpd_base / np.sqrt(60000/bpm_base / 1000)

    delta_fpdc = fpdc_drug - fpdc_base

    return {
        "FPD_baseline_ms":    round(fpd_base, 1),
        "FPD_drug_ms":        round(fpd_drug, 1),
        "FPDc_baseline_ms":   round(fpdc_base, 1),
        "FPDc_drug_ms":       round(fpdc_drug, 1),
        "delta_FPDc_ms":      round(delta_fpdc, 1),
        "BPM_baseline":       round(bpm_base, 1),
        "BPM_drug":           round(bpm_drug, 1),
        "amplitude_pct":      round(amplitude, 1),
        "arrhythmia_events":  int(arrhythmia),
        "EAD_observed":       bool(arrhythmia),
        "concentration_uM":   concentration_uM,
    }

def cipa_mea_call(mea_data: dict) -> dict:
    """
    CiPA Tier 2 call from hiPS-CM MEA data.
    Thresholds: FDA CiPA working group consensus (Gintant 2017).
    """
    delta  = mea_data.get("delta_FPDc_ms", 0)
    ead    = mea_data.get("EAD_observed", False)
    bpm_d  = mea_data.get("BPM_drug", 60)
    bpm_b  = mea_data.get("BPM_baseline", 60)
    bpm_ch = (bpm_d - bpm_b) / bpm_b * 100

    concerns = []
    if delta > 30:         concerns.append(f"ΔFPDc > 30 ms (got {delta:.1f} ms)")
    if delta > 20:         concerns.append(f"ΔFPDc > 20 ms — monitor closely")
    if ead:                concerns.append("EAD (Early After-Depolarisation) observed")
    if bpm_ch < -20:       concerns.append(f"Bradycardia (BPM change {bpm_ch:.0f}%)")

    risk = ("HIGH"     if delta > 30 or ead
       else "MODERATE" if delta > 20
       else "LOW"      if delta > 10
       else "MINIMAL")

    return {"cipa_tier2_risk": risk, "concerns": concerns,
            "recommendation": (
                "Do not progress — cardiac liability unacceptable" if risk == "HIGH"
                else "Additional CiPA Tier 3 (in vivo) required"  if risk == "MODERATE"
                else "Acceptable — proceed with monitoring"
            )}

# Compare known cardiac compounds
print("hiPS-CM MEA CiPA Tier 2 Assessment (10 μM)")
print("="*65)
cardiac_cpds = [
    ("Cisapride (HIGH QT risk)",   "COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1",  10.0),
    ("Dofetilide (HIGH QT risk)",  "CS(=O)(=O)c1ccc(CCNc2ccc(NC(=O)CCNS(C)(=O)=O)cc2)cc1", 10.0),
    ("Verapamil (moderate)",       "COc1ccc(CCN(C)CCCC(C#N)(c2ccc(OC)c(OC)c2)C(C)C)cc1OC", 10.0),
    ("Lidocaine (low risk)",       "CCN(CC)CC(=O)Nc1c(C)cccc1C",           10.0),
    ("Aspirin (minimal risk)",     "CC(=O)Oc1ccccc1C(=O)O",                10.0),
]
for name, smi, conc in cardiac_cpds:
    mea_data = simulate_hipscm_mea(smi, conc, seed=hash(name)%3000)
    call     = cipa_mea_call(mea_data)
    print(f"  {name:35s}: ΔFPDc={mea_data['delta_FPDc_ms']:+5.1f} ms  "
          f"EAD={str(mea_data['EAD_observed']):5s}  → {call['cipa_tier2_risk']}")

---
## Section 6 — Microphysiological Systems: Multi-Organ Toxicity

MPS (Microphysiological Systems) connect multiple organ chips/organoids via microfluidic flow — mimicking systemic drug distribution and organ-organ crosstalk. This directly replaces **multi-organ repeat-dose animal studies**.

In [ ]:
# ── 6.1 Multi-organ MPS scoring ───────────────────────────────────────────────
# Connected organs share media → metabolites from liver reach other organs.
# Reference: Esch 2020, Marx 2016, TissUse HUMIMIC system.

@dataclass
class MPSOrganModule:
    organ:        str
    cell_type:    str
    flow_fraction: float  # fraction of total media flow
    data:         dict = field(default_factory=dict)

class MultiOrganMPS:
    """
    Multi-organ MPS (e.g. TissUse 4-organ chip: liver–gut–kidney–brain).
    Models systemic drug distribution with hepatic first-pass metabolism.
    """
    def __init__(self, organs: list[MPSOrganModule]):
        self.organs  = {m.organ: m for m in organs}
        self.results = {}

    def run_experiment(self, smiles: str,
                        systemic_conc_uM: float,
                        days: int = 7) -> dict:
        """
        Simulate multi-organ response.
        Liver metabolises first → metabolite reaches downstream organs.
        """
        mol = Chem.MolFromSmiles(smiles) if smiles else None
        if mol is None: return {}

        # Hepatic extraction ratio (simplified)
        logp  = Descriptors.MolLogP(mol)
        clint = np.clip(logp * 5 + np.random.normal(0, 3), 1, 100)
        Qh    = 1500.0
        er    = (Qh * clint * 0.05) / (Qh + clint * 0.05)
        er    = np.clip(er, 0.05, 0.95)

        # Post-hepatic concentration (portal vein → systemic)
        post_liver_conc = systemic_conc_uM * (1 - er)

        organ_results = {}
        conc_map = {
            "liver":    systemic_conc_uM,
            "gut":      systemic_conc_uM * 3,    # higher gut exposure (oral route)
            "kidney":   post_liver_conc * 0.8,
            "brain":    post_liver_conc * 0.1,    # BBB restricts penetration
            "heart":    post_liver_conc,
        }

        for organ_name, module in self.organs.items():
            conc = conc_map.get(organ_name, post_liver_conc)
            if organ_name == "liver":
                data = simulate_liver_organoid(smiles, conc, days, seed=hash(organ_name)%999)
                risk = classify_dili_risk(data)
                organ_results[organ_name] = {**data, "risk_call": risk["DILI_risk"],
                                              "concerns": risk["concerns"]}
            elif organ_name == "heart":
                data = simulate_hipscm_mea(smiles, conc, seed=hash(organ_name)%999)
                call = cipa_mea_call(data)
                organ_results[organ_name] = {**data, "risk_call": call["cipa_tier2_risk"],
                                              "concerns": call["concerns"]}
            else:
                # Generic organ viability
                tox = np.clip(logp/6 * (np.log10(conc+1)/3), 0, 1)
                viability = max(10, 100*(1-0.8*tox) + np.random.normal(0,5))
                organ_results[organ_name] = {
                    "viability_pct": round(float(viability), 1),
                    "conc_uM":       round(float(conc), 3),
                    "risk_call":     ("HIGH" if viability < 60 else
                                      "MODERATE" if viability < 75 else "LOW"),
                }

        # Cross-organ crosstalk markers
        organ_results["_systemic"] = {
            "input_conc_uM":        systemic_conc_uM,
            "hepatic_extraction":   round(float(er), 3),
            "post_liver_conc_uM":   round(float(post_liver_conc), 3),
            "exposure_days":        days,
        }

        return organ_results

# Run 4-organ MPS for troglitazone (known multi-organ toxicant)
mps = MultiOrganMPS([
    MPSOrganModule("liver",  "iPSC-hepatocytes",    0.30),
    MPSOrganModule("gut",    "Caco-2/HT29",         0.25),
    MPSOrganModule("kidney", "Primary RPTEC",        0.25),
    MPSOrganModule("heart",  "hiPS-CM",              0.20),
])

compounds = [
    ("Troglitazone (multi-organ)", "Cc1ccc(CC2SC(=O)NC2=O)cc1OCC(C)(C)c1ccc(O)cc1"),
    ("Metformin (safe)",           "CN(C)C(=N)NC(=N)N"),
]

for cpd_name, smi in compounds:
    print(f"\n{'='*65}")
    print(f"4-Organ MPS Assessment: {cpd_name} (10 μM, 7 days)")
    print(f"{'='*65}")
    np.random.seed(42)
    results = mps.run_experiment(smi, systemic_conc_uM=10.0, days=7)

    sys_info = results.pop("_systemic", {})
    print(f"  Hepatic extraction: {sys_info.get('hepatic_extraction',0):.2f}")
    print(f"  Post-liver conc:    {sys_info.get('post_liver_conc_uM',0):.2f} μM")
    print()
    for organ, data in results.items():
        risk = data.get("risk_call", "N/A")
        icon = "⚠" if risk in ("HIGH","MODERATE") else "✓"
        conc = data.get("conc_uM", data.get("concentration_uM", "N/A"))
        print(f"  {icon} {organ:8s}: {risk:8s}  (conc={conc} μM)")

---
## Section 7 — Digital Twin Toxicology

A Digital Twin integrates PBPK, QSAR, and MPS data into a computational model of individual patient/organ response — the ultimate NAM that replaces entire animal study designs.

In [ ]:
# ── 7.1 Digital twin: PBPK + OoC + population variability ────────────────────
# Integrates:
#   PBPK (dosimetry) + OoC/organoid (biological response) + population statistics
# Reference: Wikswo 2017, Low 2021, Livingston 2020 (ALTEX)

from scipy.integrate import odeint

def pbpk_population_simulation(smiles: str,
                                 dose_mg_kg: float,
                                 n_subjects: int = 100,
                                 age_range: tuple = (20, 75),
                                 seed: int = 42) -> pd.DataFrame:
    """
    Monte Carlo PBPK simulation over a virtual population.
    Generates Cmax distribution that replaces animal dose-finding study.

    Population variability parameters (CV%):
      CLint: 40% (CYP enzyme polymorphism)
      fup:   35% (albumin variability)
      BW:    20% (body weight distribution)
      Vd:    25% (volume of distribution)
    """
    np.random.seed(seed)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return pd.DataFrame()

    logp = Descriptors.MolLogP(mol)
    mw   = Descriptors.MolWt(mol)

    # Population parameters (log-normal distributions, Table from Jamei 2009)
    BW_pop      = np.random.lognormal(np.log(70),  0.20, n_subjects)
    CLint_pop   = np.random.lognormal(np.log(10*max(0.5,logp/3)), 0.40, n_subjects)
    fup_pop     = np.clip(np.random.lognormal(np.log(0.15), 0.35, n_subjects), 0.005, 1.0)
    age_pop     = np.random.uniform(*age_range, n_subjects)

    # Age-dependent CYP3A4 activity (Edginton 2006 model)
    age_factor  = np.where(age_pop > 60, 0.75, np.where(age_pop < 30, 0.90, 1.0))
    CLint_pop   = CLint_pop * age_factor

    results = []
    for i in range(n_subjects):
        bw   = BW_pop[i]
        clint = CLint_pop[i]
        fup  = fup_pop[i]
        age  = age_pop[i]
        mppgl = 45.0
        liver_g = 1500 * (bw / 70) ** 0.85
        CLint_liver = clint * mppgl * liver_g / 1000  # L/h
        Qh = 90 * (bw / 70) ** 0.75  # allometric scaling
        CLh = (Qh * CLint_liver * fup) / (Qh + CLint_liver * fup)
        Vd  = max(0.5, (0.2 + 0.8*logp) * bw)
        k_elim = CLh / Vd
        dose_umol = (dose_mg_kg * bw * 1000 / mw) * 0.90 * 1e3  # 90% bioavail
        C0 = dose_umol / (Vd * 1000)  # μM
        t_half = np.log(2) / k_elim if k_elim > 0 else 99
        AUC = C0 / k_elim if k_elim > 0 else C0*100

        results.append({
            "subject_id": i, "age": round(age, 0), "bw_kg": round(bw, 1),
            "CLint": round(clint, 2), "fup": round(fup, 4), "CLh_L_h": round(CLh, 2),
            "Cmax_uM": round(C0, 4), "Cmax_free_uM": round(C0*fup, 4),
            "t_half_h": round(t_half, 2), "AUC_uM_h": round(AUC, 2),
        })

    return pd.DataFrame(results)

smi_diclo = "O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl"
pop_pk = pbpk_population_simulation(smi_diclo, dose_mg_kg=1.0, n_subjects=500)

print("Digital Twin: Population PBPK for Diclofenac (1 mg/kg, n=500 virtual subjects)")
print("=" * 65)
print(f"  Cmax (total):  {pop_pk['Cmax_uM'].mean():.3f} ± {pop_pk['Cmax_uM'].std():.3f} μM")
print(f"               5th–95th pct: [{pop_pk['Cmax_uM'].quantile(0.05):.3f}, "
      f"{pop_pk['Cmax_uM'].quantile(0.95):.3f}] μM")
print(f"  t½:            {pop_pk['t_half_h'].mean():.1f} ± {pop_pk['t_half_h'].std():.1f} h")
print(f"  AUC:           {pop_pk['AUC_uM_h'].mean():.1f} ± {pop_pk['AUC_uM_h'].std():.1f} μM·h")
print(f"  High-exposure (>95th pct Cmax): {(pop_pk['Cmax_uM']>pop_pk['Cmax_uM'].quantile(0.95)).sum()} subjects")

# Plot population variability
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, xlabel in [
    (axes[0], "Cmax_uM",  "Cmax (μM)"),
    (axes[1], "t_half_h", "t½ (h)"),
    (axes[2], "AUC_uM_h", "AUC (μM·h)"),
]:
    ax.hist(pop_pk[col], bins=35, color="#1565C0", alpha=0.75, edgecolor="white")
    ax.axvline(pop_pk[col].mean(),         color="#E74C3C", lw=2.5, label="Mean")
    ax.axvline(pop_pk[col].quantile(0.05), color="orange",  lw=1.8, linestyle="--", label="5th–95th")
    ax.axvline(pop_pk[col].quantile(0.95), color="orange",  lw=1.8, linestyle="--")
    ax.set_xlabel(xlabel, fontsize=11); ax.set_ylabel("Count"); ax.legend(fontsize=8)
    ax.set_title(f"Population {col}", fontweight="bold"); ax.grid(True, alpha=0.3)

plt.suptitle("Digital Twin: Population PBPK Variability (Replaces Animal Dose-Range Finding)",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

---
## Section 8 — AI Prediction with Uncertainty Quantification

Uncertainty quantification is critical for regulatory acceptance of AI/ML predictions. A confident wrong prediction is worse than an uncertain right one.

In [ ]:
# ── 8.1 Conformal prediction for regulatory toxicology ────────────────────────
# Conformal prediction gives statistically rigorous coverage guarantees.
# Reference: Venn-ABERS, IVAP, cross-conformal (Shafer & Vovk 2008)
# Regulatory relevance: FDA AI/ML framework (2021), ISO 42001

def smiles_to_ecfp4(smiles, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, n_bits)
    return np.array(fp, dtype=np.uint8)

# Simulate Ames dataset
def simulate_ames_dataset(n=1000, seed=42):
    np.random.seed(seed)
    smiles_pool = [
        "O=[N+]([O-])c1ccccc1", "Nc1ccccc1", "CC(=O)Oc1ccccc1C(=O)O",
        "Cn1cnc2c1c(=O)n(C)c(=O)n2C", "CC(=O)Nc1ccc(O)cc1",
        "c1ccc2[nH]ccc2c1", "O=Nn1ccc2ccccc21", "NCc1ccccc1",
        "OC(=O)c1ccccc1", "c1ccc2ncccc2c1",
    ]
    data = []
    for i in range(n):
        smi = smiles_pool[i % len(smiles_pool)]
        mol = Chem.MolFromSmiles(smi)
        if mol:
            has_alert = mol.HasSubstructMatch(Chem.MolFromSmarts("[N+](=O)[O-]")) or                         mol.HasSubstructMatch(Chem.MolFromSmarts("[NH2]c"))
            label = int(np.random.rand() < (0.7 if has_alert else 0.15))
            data.append({"smiles": smi, "label": label})
    return pd.DataFrame(data)

df = simulate_ames_dataset()
fps = np.array([smiles_to_ecfp4(s) for s in df.smiles])
y   = df.label.values

class ConformalPredictor:
    """
    Inductive Conformal Predictor for classification.
    Provides prediction sets with guaranteed coverage at significance level alpha.
    Reference: Vovk, Gammerman, Shafer (2005); Norinder 2014 (J Comput Aided Mol Des)
    """
    def __init__(self, model, calibration_frac=0.20):
        self.model     = model
        self.cal_frac  = calibration_frac
        self.cal_scores = None

    def calibrate(self, X_cal: np.ndarray, y_cal: np.ndarray):
        """Compute nonconformity scores on calibration set."""        probs = self.model.predict_proba(X_cal)
        # Nonconformity = 1 - P(true class)
        self.cal_scores = np.array([
            1 - probs[i, y_cal[i]] for i in range(len(y_cal))
        ])
        return self

    def predict_set(self, X_test: np.ndarray, alpha: float = 0.10) -> list:
        """
        Return prediction set (subset of {0,1}) with 1-alpha coverage guarantee.
        alpha=0.10 → 90% confidence (correct label in prediction set ≥90% of time).
        """
        probs = self.model.predict_proba(X_test)
        n_cal = len(self.cal_scores)
        sets  = []
        for prob in probs:
            pred_set = []
            for cls in range(2):
                score  = 1 - prob[cls]
                p_val  = (self.cal_scores >= score).sum() / (n_cal + 1)
                if p_val > alpha:
                    pred_set.append(cls)
            sets.append(pred_set)
        return sets

    def efficiency(self, X_test, alpha=0.10):
        """Fraction of test compounds with singleton (decisive) prediction."""        sets = self.predict_set(X_test, alpha)
        singleton = sum(1 for s in sets if len(s) == 1)
        empty     = sum(1 for s in sets if len(s) == 0)
        both      = sum(1 for s in sets if len(s) == 2)
        return {"singleton": singleton/len(sets), "empty": empty/len(sets),
                "both": both/len(sets), "n": len(sets)}

# Train + calibrate
from sklearn.model_selection import train_test_split
X_tr, X_tmp, y_tr, y_tmp = train_test_split(fps, y, test_size=0.4, stratify=y, random_state=42)
X_cal, X_te,  y_cal, y_te  = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=42)
rf.fit(X_tr, y_tr)

cp = ConformalPredictor(rf)
cp.calibrate(X_cal, y_cal)

# Evaluate coverage and efficiency
sets_test = cp.predict_set(X_te, alpha=0.10)
coverage  = np.mean([y_te[i] in sets_test[i] for i in range(len(y_te))])
eff       = cp.efficiency(X_te, alpha=0.10)

print(f"Conformal Prediction (90% confidence level):")
print(f"  Empirical coverage:   {coverage:.3f}  (guarantee ≥ 0.90 ✓)")
print(f"  Singleton rate:       {eff['singleton']:.3f}  (decisive predictions)")
print(f"  Both-class rate:      {eff['both']:.3f}  (uncertain → flag for assay)")
print(f"  Empty set:            {eff['empty']:.4f}  (anomaly — outside training)")

# Show example predictions
print(f"\nSample predictions:")
for i, (set_pred, true) in enumerate(zip(sets_test[:8], y_te[:8])):
    correct  = "✓" if true in set_pred else "✗"
    decisive = "decisive" if len(set_pred)==1 else "uncertain" if len(set_pred)==2 else "anomaly"
    print(f"  {correct} True={true}  Prediction set={set_pred}  ({decisive})")

---
## Section 9 — Statistical Power Analysis for 3Rs

Power analysis ensures you use the **minimum number of animals** necessary — a core 3Rs Reduce principle. Every animal study design requires a priori power calculation.

In [ ]:
# ── 9.1 Sample size / power calculation for 3Rs compliance ──────────────────
# ICH S7A requires power ≥ 80%; FDA expects ≥ 80% (preferably 90%).
# 3Rs principle: use the minimum n needed to achieve required power.

from scipy.stats import t, norm, ttest_ind

def power_two_sample_ttest(n: int, delta: float, sigma: float,
                            alpha: float = 0.05,
                            two_tailed: bool = True) -> float:
    """
    Statistical power for two-sample t-test.
    n      : sample size per group
    delta  : expected difference (effect size, same units as sigma)
    sigma  : pooled standard deviation (from pilot or literature)
    alpha  : type I error rate (default 0.05)
    """
    alpha_adj = alpha / 2 if two_tailed else alpha
    t_alpha   = t.ppf(1 - alpha_adj, df=2*(n-1))
    ncp       = delta / (sigma * np.sqrt(2/n))  # non-centrality parameter
    power     = 1 - t.cdf(t_alpha - ncp, df=2*(n-1)) + t.cdf(-t_alpha - ncp, df=2*(n-1))
    return float(power)

def find_min_n(delta: float, sigma: float,
               target_power: float = 0.80,
               alpha: float = 0.05) -> dict:
    """
    Find minimum n per group to achieve target power.
    Iterates n from 2 upward — stops at target.
    """
    for n in range(2, 500):
        pwr = power_two_sample_ttest(n, delta, sigma, alpha)
        if pwr >= target_power:
            return {"n_per_group": n, "n_total": 2*n, "power": round(pwr, 4),
                    "alpha": alpha, "delta": delta, "sigma": sigma,
                    "effect_size_d": round(delta/sigma, 3)}
    return {"n_per_group": 500, "power": None}

# ── 3Rs power analysis scenarios ──────────────────────────────────────────────
print("3Rs Power Analysis: Minimum Animal Numbers for Common Tox Studies")
print("="*70)
scenarios = [
    # (study_name, expected_delta, sigma, alpha, power_target, traditional_n)
    ("Rodent 28-day (body wt, 10% diff)", 10.0, 15.0, 0.05, 0.80, 10),
    ("Liver weight (15% diff, high σ)",   15.0, 25.0, 0.05, 0.80, 10),
    ("Clinical chemistry (ALT, 2-fold)",   50.0, 40.0, 0.05, 0.80,  8),
    ("Developmental weight (5% diff)",      5.0, 10.0, 0.05, 0.80, 20),
    ("High-precision (90% power)",         10.0, 15.0, 0.05, 0.90, 12),
]
print(f"{'Study':40s} {'Min n/grp':>9} {'Total':>7} {'Power':>7} {'d':>5} {'Traditional':>12}")
print("-"*80)
total_required = 0
total_traditional = 0
for study, delta, sigma, alpha, power_tgt, trad_n in scenarios:
    result = find_min_n(delta, sigma, power_tgt, alpha)
    n      = result["n_per_group"]
    saving = trad_n - n
    total_required    += result["n_total"]
    total_traditional += trad_n * 2
    flag = " ← REDUCE" if saving > 0 else ""
    print(f"{study:40s} {n:>9}  {result['n_total']:>6}  {result['power']:>6.3f}  "
          f"{result['effect_size_d']:>5.2f}  {trad_n*2:>8}{flag}")

savings_pct = (total_traditional - total_required) / total_traditional * 100
print(f"\nTotal animals required:    {total_required}")
print(f"Traditional animal count:  {total_traditional}")
print(f"3Rs reduction:             {total_traditional - total_required} animals "
      f"({savings_pct:.0f}% reduction)
")

# ── Dose-spacing optimisation for maximum info with fewer dose groups ──────────
print("Optimal Dose Spacing (D-optimal design for dose-response)")
print("=" * 60)
print(
    "Traditional: equal log-spacing (e.g. 1, 3, 10, 30, 100 mg/kg)
"
    "D-optimal:   place doses near EC10, EC50, EC90 -- 3 groups gives
"
    "             same information as 5 equal-log groups -- 40% fewer animals
"
    "
BMD preferred over NOAEL for 3Rs:
"
    "  NOAEL: requires negative dose group (wastes animals at no-effect)
"
    "  BMD:   models entire curve -- fewer dose groups needed
"
    "  BMD10: dose at 10% benchmark response = NOAEL equivalent
"
    "  BMDL:  lower 95pct CI on BMD (used in risk assessment)"
)

---
## Section 10 — Regulatory IATA Report (3Rs-Focused)

The complete 3Rs-justification package for a regulatory submission. Demonstrates how to assemble NAM evidence into a structured IATA WoE report that argues **no new animal experiments are needed**.

In [ ]:
# ── 10.1 Complete 3Rs IATA report generator ──────────────────────────────────
# Follows: OECD GD 255, ECHA 3Rs guidance, FDA Modernization Act 2.0 framework

def generate_3rs_report(smiles: str, compound_name: str,
                          intended_use: str = "pharmaceutical") -> dict:
    """
    Generate a 3Rs-justification IATA report.
    Returns structured dict suitable for regulatory submission.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {"error": "Invalid SMILES"}

    np.random.seed(42)
    mw    = Descriptors.MolWt(mol)
    logp  = Descriptors.MolLogP(mol)
    tpsa  = Descriptors.TPSA(mol)

    # ── Step 1: Data gap analysis ──────────────────────────────────────────────
    required_endpoints = {
        "pharmaceutical": [
            "Acute toxicity (LD50 equivalent)",
            "Genotoxicity (in vitro)",
            "Cardiac safety (hERG/QT)",
            "Hepatotoxicity",
            "Skin sensitisation",
            "Reproductive/developmental toxicity",
        ]
    }.get(intended_use, [])

    # ── Step 2: Existing data assessment ──────────────────────────────────────
    existing_data_sources = [
        {"db": "ToxCast",  "n_assays": 9000, "checked": True,  "relevant_hits": 3},
        {"db": "ChEMBL",   "n_assays": 1400, "checked": True,  "relevant_hits": 1},
        {"db": "DILIrank", "n_cpds": 1036,   "checked": True,  "match": False},
        {"db": "DrugMatrix","n_cpds": 600,    "checked": True,  "match": False},
    ]

    # ── Step 3: NAM assessment per endpoint ────────────────────────────────────
    ooc_data    = simulate_ooc_experiment(smiles, "liver", 10.0, seed=1)
    ooc_score   = score_ooc_toxicity(ooc_data)
    mea_data    = simulate_hipscm_mea(smiles, 10.0, seed=2)
    mea_call    = cipa_mea_call(mea_data)
    organ_data  = simulate_liver_organoid(smiles, 10.0, 14, seed=3)
    dili_risk   = classify_dili_risk(organ_data)
    skin_result = {
        "DPRA": "NEGATIVE", "KeratinoSens": "NEGATIVE",
        "hCLAT": "NEGATIVE", "hazard_call": "NON-SENSITISER"
    }
    fp      = smiles_to_ecfp4(smiles)
    in_ad   = False
    qsar_prob = 0.12

    # ── Step 4: Decision table per endpoint ────────────────────────────────────
    endpoint_decisions = {
        "Acute toxicity": {
            "NAM_used":     "QSAR LD50 (ECOSAR) + PBPK IVIVE",
            "data_source":  "Computational prediction within AD",
            "call":         "No concern (predicted LD50 > 2000 mg/kg)",
            "animal_needed":False,
            "justification":"QSAR within AD + IVIVE dose estimate sufficient for Ro5-compliant oral drug",
            "guideline":    "FDA Modernization Act 2.0, OECD QSAR GD 69",
        },
        "Genotoxicity": {
            "NAM_used":     "ICH M7 two-method (SA + QSAR)",
            "data_source":  "SMARTS alerts + RF-ECFP4 model",
            "call":         "No structural alert; QSAR probability = 0.12 (negative)",
            "animal_needed":False,
            "justification":"Both in silico methods negative — ICH M7 Class 5, no further testing",
            "guideline":    "ICH M7(R2) 2023",
        },
        "Cardiac safety": {
            "NAM_used":     f"CiPA: multi-channel IC50 + hiPS-CM MEA",
            "data_source":  "CiPA Tier 1+2 assays",
            "call":         f"ΔFPDc = {mea_data.get('delta_FPDc_ms',0):.1f} ms — {mea_call['cipa_tier2_risk']} risk",
            "animal_needed": mea_call["cipa_tier2_risk"] in ("HIGH","MODERATE"),
            "justification":"CiPA replaces hERG-only + in vivo QT per ICH E14/S7B 2022",
            "guideline":    "ICH E14/S7B 2022; FDA CiPA Initiative",
        },
        "Hepatotoxicity": {
            "NAM_used":     "Liver organoid (3D iPSC spheroids, 14-day exposure)",
            "data_source":  "InSphero GravityPLUS",
            "call":         f"DILI risk: {dili_risk['DILI_risk']} ({dili_risk['n_concerns']} concerns)",
            "animal_needed": dili_risk["DILI_risk"] == "HIGH",
            "justification":"3D liver organoid ≥85% concordance with in vivo (Proctor 2017)",
            "guideline":    "OECD GD 255; ECHA 3Rs guidance; FDA Modernization Act 2.0",
        },
        "Skin sensitisation": {
            "NAM_used":     "OECD TG 497 DA2 (2o3): DPRA + KeratinoSens + h-CLAT",
            "data_source":  "In vitro defined approach",
            "call":         f"NON-SENSITISER (0/3 methods positive)",
            "animal_needed":False,
            "justification":"Validated OECD DA — no animal test required per TG 497",
            "guideline":    "OECD TG 442C/D/E + TG 497 (2023)",
        },
    }

    # ── Step 5: Overall 3Rs conclusion ─────────────────────────────────────────
    n_animal_needed = sum(1 for d in endpoint_decisions.values() if d["animal_needed"])
    n_total         = len(endpoint_decisions)

    report = {
        "report_type":       "3Rs NAM Justification Package",
        "regulatory_framework": [
            "FDA Modernization Act 2.0 (2022)",
            "OECD GD 255 IATA",
            "ICH M7(R2), E14/S7B 2022",
            "OECD TG 497 (2023)",
            "EU Directive 2010/63/EU",
        ],
        "compound": {
            "name": compound_name, "smiles": smiles,
            "MW": round(mw,1), "LogP": round(logp,2),
        },
        "3rs_summary": {
            "Replace": f"{n_total - n_animal_needed}/{n_total} endpoints covered by NAMs",
            "Reduce":  "Power analysis reduces n/group by 20-40% vs traditional design",
            "Refine":  "Biomarker endpoints replace histopathology where possible",
            "net_animal_studies_required": n_animal_needed,
            "net_animal_studies_avoided":  n_total - n_animal_needed,
        },
        "endpoint_decisions": endpoint_decisions,
        "recommendation": (
            "No new animal experiments required" if n_animal_needed == 0
            else f"{n_animal_needed} endpoint(s) require targeted animal follow-up"
        ),
    }
    return report

report = generate_3rs_report("CC(=O)Oc1ccccc1C(=O)O", "Aspirin", "pharmaceutical")

print("3Rs NAM JUSTIFICATION PACKAGE")
print("="*65)
print(f"Compound: {report['compound']['name']}")
print(f"\n3Rs Summary:")
for k, v in report["3rs_summary"].items():
    print(f"  {k:35s}: {v}")
print(f"\nEndpoint decisions:")
for ep, dec in report["endpoint_decisions"].items():
    flag = "✓ No animal" if not dec["animal_needed"] else "⚠ Animal needed"
    print(f"  {flag:15s} {ep:25s}: {dec['call'][:45]}")
print(f"\n→ CONCLUSION: {report['recommendation']}")

In [ ]:
# ── 10.2 Regulatory cheatsheet ───────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║        Animal-Free Toxicology: 3Rs & NAMs — Regulatory Reference        ║
╠══════════════════════════════════════════════════════════════════════════╣
║ ORGAN-ON-CHIP / ORGANOID ENDPOINTS                                       ║
║  Liver OoC     ATP, LDH, albumin, urea, CYP3A4, ALT/AST, ROS, BSEP    ║
║  Liver organoid steatosis score, apoptosis, CDFDA transport            ║
║  hiPS-CM MEA   ΔFPDc (>20 ms flag, >30 ms = HIGH risk), EAD, BPM      ║
║  Gut OoC       TEER (>20% drop = barrier loss), FD4 permeability       ║
║  Kidney OoC    KIM-1, NGAL, creatinine clearance, OAT transporter      ║
║  Brain organoid MEA bursting, synchrony, apoptosis, neuronal markers   ║
╠══════════════════════════════════════════════════════════════════════════╣
║ KEY PLATFORMS (commercial)                                               ║
║  Emulate Bio       Lung/liver/gut chips — FDA MOU partner               ║
║  CN Bio            PhysioMimix liver OoC — most validated for DILI     ║
║  InSphero          3D liver spheroids — GravityPLUS, FDA accepted       ║
║  Mimetas           OrganoPlate gut/kidney — 96-well throughput         ║
║  TissUse           HUMIMIC 4-organ connected MPS                       ║
║  Ncardia           Cor.4U hiPS-CM — validated for CiPA Tier 2          ║
║  Axion Biosystems  Maestro MEA — cardiac + neuronal electrophysiology  ║
╠══════════════════════════════════════════════════════════════════════════╣
║ REGULATORY ACCEPTANCE (2024 status)                                      ║
║  Genotoxicity     ICH M7(R2): in silico REPLACES animal for impurities ║
║  Skin sensitisation OECD TG 497 DA2: fully replaces LLNA (2023)       ║
║  Cardiac safety   ICH E14/S7B 2022: CiPA replaces hERG-only           ║
║  Hepatotoxicity   FDA pilot: 3D organoid data accepted in NDA (2022+)  ║
║  Acute toxicity   EPA: QSAR accepted for Tier 1 hazard (TSCA 2016)    ║
║  Repeat dose      Under validation: OoC/MPS (OECD WNT, 2024–2026)    ║
╠══════════════════════════════════════════════════════════════════════════╣
║ DATABASES (check before running any new experiment)                      ║
║  CompTox/ToxCast  ~10,000 chemicals, 9,000 HTS assays (EPA)           ║
║  Tox21            12,000 chemicals, 71 assays (NIH/EPA/FDA)            ║
║  Open TG-GATEs    170 compounds, rat + human hepatocyte transcriptomics║
║  DrugMatrix       600 compounds, rat organ toxicogenomics              ║
║  DILIrank         1,036 drugs, FDA DILI severity classification        ║
║  ECOTOX           13,000 chemicals, aquatic + terrestrial ecotox       ║
╠══════════════════════════════════════════════════════════════════════════╣
║ 3Rs DESIGN PRINCIPLES                                                    ║
║  Replace:  Use NAM data before ANY animal study — check DB first       ║
║  Reduce:   Power analysis (≥80% power, minimum n) — ALWAYS required    ║
║  Refine:   BMD > NOAEL (fewer dose groups), biomarker endpoints        ║
║  Report:   ARRIVE guidelines 2.0 for animal experiments               ║
╠══════════════════════════════════════════════════════════════════════════╣
║ UNCERTAINTY MANAGEMENT                                                   ║
║  Conformal prediction  guaranteed coverage at α, flags uncertain cpds  ║
║  MC PBPK               population variability → Cmax 5th–95th pct     ║
║  Cytotox correction    burst ratio removes artefacts from HTS          ║
║  AD flagging           out-of-domain → do not use prediction           ║
╚══════════════════════════════════════════════════════════════════════════╝
""")